# NB_SLV_POI_AMENITY

RECOMPUTE `lh_vv_silver.dbo.slv_poi_amenity`. Key cờ google khớp `source_flags_json` = true; key tách từ `types[] =` khớp đúng một phần tử `types_json`. `json_type = 'a'` và `json_type in (...)` khớp một phần tử trong danh sách, cùng `types_json` của `slv_poi_entity_non_enrich`. `business_sector` từ `slv_poi_entity_non_enrich`.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
# BEGIN_AMENITY_MATCH
import re

FLAG_KEYS = ("servesBreakfast", "servesBrunch", "servesLunch", "servesDinner", "servesBeer", "servesWine",
             "servesCocktails", "servesDessert", "servesCoffee", "servesVegetarianFood", "allowsDogs",
             "goodForChildren", "goodForGroups", "goodForWatchingSports", "liveMusic", "menuForChildren",
             "outdoorSeating", "restroom", "reservable", "takeout", "delivery", "dineIn", "curbsidePickup")
OBJECT_KEYS = ("parkingOptions", "paymentOptions", "accessibilityOptions", "evChargeOptions", "fuelOptions")
TYPES_PATH = "types[]"
# Key cờ (seed "key = true" bỏ hậu tố): khớp source_flags_json = true. Khoá (business_sector, amenity_group, amenity_raw).
FLAG_MATCH = frozenset({
    ("accommodation", "general_amenities", "parkingOptions.freeStreetParking"),
    ("accommodation", "general_amenities", "parkingOptions.valetParking"),
    ("accommodation", "general_amenities", "parkingOptions.freeGarageParking"),
    ("accommodation", "general_amenities", "parkingOptions.paidGarageParking"),
    ("accommodation", "general_amenities", "outdoorSeating"),
    ("accommodation", "general_amenities", "restroom"),
    ("accommodation", "general_amenities", "goodForChildren"),
    ("accommodation", "general_amenities", "goodForGroups"),
    ("accommodation", "general_amenities", "liveMusic"),
    ("accommodation", "general_amenities", "parkingOptions.freeParkingLot"),
    ("accommodation", "general_amenities", "parkingOptions.paidParkingLot"),
    ("accommodation", "general_amenities", "servesCocktails"),
    ("accommodation", "general_amenities", "parkingOptions.freeElectricVehicleParking"),
    ("accommodation", "general_amenities", "parkingOptions.paidElectricVehicleParking"),
    ("accommodation", "accessibility", "accessibilityOptions.wheelchairAccessibleSeating"),
    ("accommodation", "accessibility", "accessibilityOptions.wheelchairAccessibleElevator"),
    ("accommodation", "accessibility", "accessibilityOptions.wheelchairAccessibleEntrance"),
    ("accommodation", "accessibility", "accessibilityOptions.wheelchairAccessibleRestroom"),
    ("accommodation", "accessibility", "accessibilityOptions.wheelchairAccessibleParking"),
    ("f_n_b", "space_and_services", "restroom"),
    ("f_n_b", "space_and_services", "goodForChildren"),
    ("f_n_b", "space_and_services", "accessibilityOptions.wheelchairAccessibleEntrance"),
    ("f_n_b", "space_and_services", "goodForWatchingSports"),
    ("f_n_b", "space_and_services", "liveMusic"),
    ("f_n_b", "space_and_services", "dineIn"),
    ("attraction", "facilities_and_services", "parkingOptions.freeParkingLot"),
    ("attraction", "facilities_and_services", "parkingOptions.paidParkingLot"),
})
# Key tách từ seed "types[] = a, b": khớp đúng một phần tử slv_poi_entity_non_enrich.types_json.
TYPE_MATCH = frozenset({
    ("accommodation", "general_amenities", "swimming_pool"),
    ("accommodation", "general_amenities", "hot_spring"),
    ("accommodation", "general_amenities", "sauna"),
    ("accommodation", "general_amenities", "garden"),
    ("accommodation", "general_amenities", "botanical_garden"),
    ("accommodation", "general_amenities", "meeting_rooms"),
    ("f_n_b", "specialty_tags", "food_stand"),
    ("f_n_b", "specialty_tags", "street_food_gathering"),
    ("f_n_b", "specialty_tags", "seafood_restaurant"),
    ("f_n_b", "specialty_tags", "oyster_bar_restaurant"),
    ("f_n_b", "specialty_tags", "steakhouse"),
    ("f_n_b", "specialty_tags", "bakery"),
    ("f_n_b", "specialty_tags", "pastry_shop"),
    ("f_n_b", "specialty_tags", "cake_shop"),
})
# Câu types[] có/thuộc đã chuẩn hóa: json_type = một token, hoặc json_type in nhiều token.
JSON_EQ_RE = re.compile(r"json_type = '([A-Za-z0-9_]+)'")
JSON_IN_RE = re.compile(r"json_type in \('([A-Za-z0-9_]+(?:', '[A-Za-z0-9_]+)*)'\)")
JSON_MATCH = frozenset({
    ("accommodation", "general_amenities", "json_type in ('vietnamese_restaurant', 'chinese_restaurant', 'japanese_restaurant', 'korean_restaurant', 'thai_restaurant', 'french_restaurant', 'italian_restaurant', 'pizza_restaurant', 'american_restaurant', 'european_restaurant', 'british_restaurant', 'german_restaurant', 'spanish_restaurant', 'greek_restaurant', 'steak_house', 'portuguese_restaurant', 'austrian_restaurant', 'bavarian_restaurant', 'belgian_restaurant', 'danish_restaurant', 'dutch_restaurant', 'scandinavian_restaurant', 'swiss_restaurant', 'polish_restaurant', 'romanian_restaurant', 'russian_restaurant', 'ukrainian_restaurant', 'croatian_restaurant', 'czech_restaurant', 'hungarian_restaurant', 'australian_restaurant', 'californian_restaurant', 'tex_mex_restaurant', 'fish_and_chips_restaurant', 'basque_restaurant', 'irish_restaurant', 'soul_food_restaurant', 'southwestern_us_restaurant', 'eastern_european_restaurant', 'indian_restaurant', 'north_indian_restaurant', 'south_indian_restaurant', 'indonesian_restaurant', 'malaysian_restaurant', 'filipino_restaurant', 'cambodian_restaurant', 'burmese_restaurant', 'tibetan_restaurant', 'taiwanese_restaurant', 'afghani_restaurant', 'african_restaurant', 'ethiopian_restaurant', 'moroccan_restaurant', 'lebanese_restaurant', 'israeli_restaurant', 'persian_restaurant', 'turkish_restaurant', 'middle_eastern_restaurant', 'mediterranean_restaurant', 'mexican_restaurant', 'latin_american_restaurant', 'south_american_restaurant', 'brazilian_restaurant', 'argentinian_restaurant', 'colombian_restaurant', 'chilean_restaurant', 'peruvian_restaurant', 'cuban_restaurant', 'caribbean_restaurant', 'hawaiian_restaurant', 'asian_restaurant', 'asian_fusion_restaurant', 'mongolian_barbecue_restaurant', 'pakistani_restaurant', 'bangladeshi_restaurant', 'sri_lankan_restaurant', 'fusion_restaurant', 'bistro', 'bakery', 'cake_shop', 'pastry_shop', 'dessert_restaurant', 'dessert_shop', 'donut_shop', 'ice_cream_shop', 'confectionery', 'chocolate_shop', 'bar', 'pub', 'night_club', 'cocktail_bar', 'wine_bar', 'lounge_bar', 'sports_bar', 'hookah_bar', 'irish_pub', 'beer_garden', 'karaoke', 'fast_food_restaurant', 'meal_takeaway', 'sandwich_shop', 'deli', 'snack_bar', 'burrito_restaurant', 'taco_restaurant', 'chicken_restaurant', 'kebab_shop', 'gyro_restaurant', 'shawarma_restaurant', 'hamburger_restaurant', 'vegan_restaurant', 'vegetarian_restaurant', 'seafood_restaurant', 'oyster_bar_restaurant', 'coffee_shop', 'cafe', 'coffee_roastery', 'coffee_stand', 'tea_house', 'juice_shop', 'hot_pot_restaurant', 'barbecue_restaurant', 'bar_and_grill', 'korean_barbecue_restaurant', 'noodle_shop', 'ramen_restaurant', 'chinese_noodle_restaurant', 'fine_dining_restaurant', 'buffet_restaurant', 'brunch_restaurant', 'breakfast_restaurant', 'halal_restaurant', 'gastropub', 'brewpub', 'brewery', 'winery', 'restaurant', 'food_court', 'cafeteria', 'diner')"),
    ("accommodation", "general_amenities", "json_type in ('bar', 'pub', 'night_club', 'cocktail_bar', 'wine_bar', 'lounge_bar', 'sports_bar', 'hookah_bar', 'irish_pub', 'beer_garden', 'karaoke')"),
    ("accommodation", "general_amenities", "json_type in ('spa', 'day_spa', 'wellness_center', 'massage', 'skin_care_clinic', 'acupuncture_clinic', 'ayurvedic_clinic', 'hair_salon', 'hair_care', 'nail_salon', 'beauty_salon', 'barber_shop')"),
    ("accommodation", "general_amenities", "json_type in ('gym', 'fitness_center', 'sports_club', 'recreation_center', 'yoga_studio', 'pilates_studio')"),
    ("accommodation", "general_amenities", "json_type = 'electric_vehicle_charging_station'"),
    ("attraction", "facilities_and_services", "json_type in ('historical_landmark', 'archaeological_site', 'ruins', 'castle', 'palace', 'monument', 'sculpture', 'lighthouse')"),
    ("attraction", "facilities_and_services", "json_type = 'art_gallery'"),
    ("attraction", "facilities_and_services", "json_type = 'planetarium'"),
    ("attraction", "facilities_and_services", "json_type in ('church', 'hindu_temple', 'mosque', 'synagogue', 'buddhist_temple', 'place_of_worship')"),
    ("attraction", "facilities_and_services", "json_type in ('restaurant', 'cafe', 'coffee_shop')"),
    ("attraction", "facilities_and_services", "json_type = 'gift_shop'"),
    ("f_n_b", "specialty_tags", "json_type in ('bar', 'pub', 'night_club', 'cocktail_bar', 'wine_bar', 'lounge_bar', 'sports_bar', 'hookah_bar', 'irish_pub', 'beer_garden', 'karaoke')"),
})
LEGACY_RE = re.compile(r"[A-Za-z][A-Za-z0-9]*(?:\.[A-Za-z][A-Za-z0-9]*)? = true|types\[\] = .+")


def resolve_condition(sector, group, raw):
    """(sector, group, amenity_raw) -> (kind, path, giá trị) hoặc None.
    Ví dụ ("accommodation", "general_amenities", "outdoorSeating") -> ("flag", "flags.outdoorSeating", "true").
    json_type in trả kind json_in, giá trị là các token nối bằng dấu phẩy."""
    key = (sector, group, (raw or "").strip())
    if key in TYPE_MATCH:
        return ("type", TYPES_PATH, key[2])
    if key in JSON_MATCH:
        one = JSON_EQ_RE.fullmatch(key[2])
        if one:
            return ("type", TYPES_PATH, one.group(1))
        many = JSON_IN_RE.fullmatch(key[2])
        if many:
            return ("json_in", TYPES_PATH, many.group(1).replace("', '", ","))
        return None
    if key not in FLAG_MATCH:
        return None
    head, _, leaf = key[2].partition(".")
    if not leaf:
        return ("flag", f"flags.{head}", "true") if head in FLAG_KEYS else None
    return ("flag", f"{head}.{leaf}", "true") if head in OBJECT_KEYS else None


def is_legacy_form(raw):
    """amenity_raw còn dạng cũ (key = true hoặc types[] = ...): seed chưa chạy lại full."""
    return LEGACY_RE.fullmatch((raw or "").strip()) is not None


def match_conditions(rows):
    """rows: cột từ điển google active, có thể kèm service_flags ở cuối.
    Điều kiện = các cột đó cộng (kind, path, value) ở đuôi. Khoá 3 cột trùng ; số dòng dạng cũ."""
    conds = [tuple(r) + hit for r in rows if (hit := resolve_condition(r[0], r[1], r[2]))]
    seen = {}
    for c in conds:
        seen[c[:3]] = seen.get(c[:3], 0) + 1
    dup = sorted(k for k, n in seen.items() if n > 1)
    return conds, dup, sum(is_legacy_form(r[2]) for r in rows)


def legacy_blocks_write(legacy):
    """amenity_raw dạng cũ còn trong từ điển thì không được ghi."""
    return legacy > 0

def match_coverage(conds):
    """Tập đã resolve phải bằng FLAG_MATCH | TYPE_MATCH | JSON_MATCH. Trả (thiếu, thừa)."""
    found = {c[:3] for c in conds}
    expected = FLAG_MATCH | TYPE_MATCH | JSON_MATCH
    return sorted(expected - found), sorted(found - expected)

# END_AMENITY_MATCH


In [ ]:
GOOGLE = "google"
COND_SCHEMA = ("business_sector string, amenity_group string, amenity_raw string, amenity string, "
               "amenity_vi string, service_flags string, _kind string, _path string, _v string")
OUT_COLS = ("poi_id", "business_sector", "amenity_group", "amenity_raw", "amenity", "amenity_vi", "service_flags")


def build(ctx: NodeContext) -> DataFrame:
    d = ctx.input("dict")
    if dict(d.dtypes).get("service_flags") != "string":
        raise ConfigError("ref_dictionary_amenities thiếu cột service_flags STRING: chạy cell dictseedpy của NB_CREATE_DDL")
    # D10: service_flags đi cùng dòng điều kiện đã khớp; không join từ điển lần hai.
    rows = [tuple(r) for r in (d
            .where((F.lower(F.trim("source_name")) == GOOGLE) & (F.col("is_active") == F.lit(True)))
            .select("business_sector", "amenity_group", "amenity_raw", "amenity", "amenity_vi", "service_flags").collect())]
    conds, dup, legacy = match_conditions(rows)
    if dup:
        raise QualityGateError(f"ref_dictionary_amenities: key google trùng trong (business_sector, amenity_group) {dup}")
    missing, extra = match_coverage(conds)
    if missing or extra:
        raise QualityGateError(
            "ref_dictionary_amenities: key google không khớp FLAG_MATCH | TYPE_MATCH | JSON_MATCH "
            f"thiếu={missing} thừa={extra}"
        )
    if legacy_blocks_write(legacy):
        raise QualityGateError(f"ref_dictionary_amenities: còn {legacy} amenity_raw dạng cũ; không ghi")
    cond = F.broadcast(spark.createDataFrame(conds, COND_SCHEMA))
    # source_flags_json = {"flags": {...}, "parkingOptions": {...} | null, ...}; object null -> không nổ dòng.
    src = ctx.input("amenity").select("poi_id", "source_flags_json")
    top = src.select("poi_id", F.explode(F.from_json("source_flags_json", "map<string,string>")).alias("_top", "_obj"))
    flat = (top.select("poi_id", "_top", F.explode(F.from_json("_obj", "map<string,string>")).alias("_leaf", "_v"))
               .select("poi_id", F.concat_ws(".", "_top", "_leaf").alias("_path"), "_v"))
    ent = ctx.input("entity").select("poi_id", "business_sector", "source_name", "types_json")
    flags = flat.join(ent.select("poi_id", "business_sector"), "poi_id").select("poi_id", "business_sector", "_path", "_v")
    # types_json = ["bakery", "cafe", ...] của POI google; phần tử lặp chỉ tính 1 lần.
    types = (ent.where(F.lower(F.trim("source_name")) == GOOGLE)
                .select("poi_id", "business_sector", F.lit(TYPES_PATH).alias("_path"),
                        F.explode(F.from_json("types_json", "array<string>")).alias("_v"))
                .dropDuplicates(["poi_id", "_v"]))
    facts = flags.unionByName(types)
    simple = cond.where(F.col("_kind") != F.lit("json_in"))
    listed = cond.where(F.col("_kind") == F.lit("json_in")).withColumnRenamed("_v", "_allow")
    out_one = facts.join(simple, ["business_sector", "_path", "_v"])
    out_in = (facts.join(listed, ["business_sector", "_path"])
                   .where(F.array_contains(F.split(F.col("_allow"), ","), F.col("_v")))
                   .drop("_allow"))
    out = out_one.unionByName(out_in).dropDuplicates(list(OUT_COLS) + ["_kind"])
    src_rows = src.count()
    unparsed = src.where(F.col("source_flags_json").isNotNull()
                         & F.from_json("source_flags_json", "map<string,string>").isNull()).count()
    joined = src.join(ent.select("poi_id"), "poi_id").count()
    by_kind = {r["_kind"]: r["count"] for r in out.groupBy("_kind").count().collect()}
    n = sum(by_kind.values())
    n_flag = sum(c[-3] == "flag" for c in conds)
    ctx.metrics.update(src_rows=src_rows, pois_joined=joined, pois_without_entity=src_rows - joined,
                       unparsed_json=unparsed, dict_google_active=len(rows), dict_legacy_form=legacy,
                       conditions_matchable=len(conds), conditions_flag=n_flag, conditions_type=len(conds) - n_flag,
                       conditions_service_flags=sum(c[5] is not None for c in conds),
                       conditions_skipped=len(rows) - len(conds),
                       rows_flag=by_kind.get("flag", 0),
                       rows_type=by_kind.get("type", 0) + by_kind.get("json_in", 0), rows=n)
    # Không có dòng khớp là trạng thái hợp lệ: DataFrame rỗng để RECOMPUTE soft-delete amenity cũ.
    return out.select(*OUT_COLS)


SPEC = NodeSpec(
    trg_tbl="slv_poi_amenity",
    inputs={"amenity": "slv_3p_poi_amenity", "entity": "slv_poi_entity_non_enrich", "dict": "ref_dictionary_amenities"},
    pk=("poi_id", "business_sector", "amenity_group", "amenity_raw"),
    build=build,
    required_inputs=("amenity", "entity", "dict"),
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
